# MagFlow Data Processing Pipeline

This notebook covers the complete workflow for processing magnetometer data from raw `.DAT` (Aidu) formats to structured CSVs, and creating interactive 3D/2D spatial visualizations delimited by coordinate boundaries.

## 1. Data Conversion

Convert the raw Aidu `.DAT` files into a single, standardized CSV file. The process includes:
- Correcting encoding formats.
- Removing duplicates headers and resolving repeated columns.
- Converting Longitude/Latitude from DMS to decimal degrees.

In [1]:
# To convert dataset from .DAT file(s) to a readable .CSV file
# The .DAT file(s) should be in the 'rawdata' folder, and the output .CSV file will be saved in the same folder as this script

import os
import pandas as pd
import chardet

def convert_dat_to_csv(input_folder, output_file, selected_files=None):
    """
    Convert .DAT files in the specified folder to a single .CSV file.

    Parameters:
        input_folder (str): Path to the folder containing .DAT files.
        output_file (str): Path for the output .CSV file.
        selected_files (list, optional): List of specific .DAT files to process.
                                         If None, all files in the folder are processed.
    """

    # Dictionary to translate Chinese header names to English.
    header_translation = {
        "文件号": "File Number",
        "测点增量": "Measurement Increment",
        "数据": "Data",
        "标记": "Marker",
        "测量时间": "Measurement Time",
        "坐标纬度": "Latitude",
        "坐标经度": "Longitude",
        "仪器号": "Instrument Number",
        "测区号": "Measurement Area Number",
        "操作员": "Operator",
        "方向": "Direction",
        "线距": "Line Distance",
        "点距": "Point Distance",
        "配谐": "Harmonic",
        "模式": "Mode",
        "预置场": "Preset Field",
        "采样间隔": "Sampling Interval"
    }

    # Helper function to convert coordinates in DMS (Degrees:Minutes/Seconds) format to decimal degrees.
    def dms_to_decimal(dms_string):
        # Split the string at the colon.
        parts = dms_string.split(":")
        # The last character of the first part represents the direction (e.g., N, S, E, W).
        direction = parts[0][-1]
        # The second part contains the numeric values separated by "/".
        dms_values = parts[1].split("/")

        # Convert the degrees, minutes, and seconds parts to floats.
        degrees = float(dms_values[0])
        minutes = float(dms_values[1])
        seconds = float(dms_values[2])
        # Convert the DMS values to a decimal degree.
        decimal_degrees = degrees + (minutes / 60) + (seconds / 3600)

        # If the direction is South or West, the decimal degree should be negative.
        if direction in ["S", "W"]:
            decimal_degrees *= -1

        return decimal_degrees

    # Initialize a list to hold data from each file.
    all_data = []

    # Loop through all files in the input folder.
    for file_name in os.listdir(input_folder):
        # Process only files with the .DAT extension and, if specified, files in the selected_files list.
        if file_name.endswith(".DAT") and (selected_files is None or file_name in selected_files):
            # Construct the full file path.
            file_path = os.path.join(input_folder, file_name)

            # Detect the file encoding using chardet.
            with open(file_path, 'rb') as file:
                result = chardet.detect(file.read())
                encoding = result['encoding']

            # Read the .DAT file into a DataFrame.
            # Assumes that the file is tab-separated.
            df = pd.read_csv(file_path, encoding=encoding, sep='\t')

            # Remove any rows that are duplicate headers.
            # This compares each row with the header row and removes it if they match.
            header_row = df.columns
            df = df[~df.apply(lambda row: (row == header_row).all(), axis=1)]

            # Rename the columns using the header_translation dictionary.
            df.rename(columns=header_translation, inplace=True)

            # Append the processed DataFrame to the list.
            all_data.append(df)

    # Concatenate all individual DataFrames into a single DataFrame.
    combined_df = pd.concat(all_data, ignore_index=True)

    # Remove duplicate rows with the same measurement time
    combined_df.drop_duplicates(subset=["Measurement Time"], inplace=True)

    # Convert Latitude and Longitude from DMS to decimal degrees if they exist.
    if "Latitude" in combined_df.columns and "Longitude" in combined_df.columns:
        combined_df["Latitude"] = combined_df["Latitude"].apply(dms_to_decimal)
        combined_df["Longitude"] = combined_df["Longitude"].apply(dms_to_decimal)
        # Remove rows where either Latitude or Longitude is 0.
        combined_df = combined_df[(combined_df["Latitude"] != 0) & (combined_df["Longitude"] != 0)]

    # Check for duplicate column names in the final DataFrame.
    duplicate_columns = combined_df.columns[combined_df.columns.duplicated()].tolist()
    if duplicate_columns:
        print(f"Warning: The following columns are repeated in the output: {duplicate_columns}")
    else:
        print("No duplicate columns found in the output.")

    # Save the converted data as a .CSV file
    combined_df.to_csv(output_file, index=False, encoding="utf-8")

    # Provide a success message indicating which files were processed and where the output was saved.
    if selected_files:
        print(f"Data from {', '.join(selected_files)} is successfully converted and saved to {output_file}.")
    else:
        print(f"Data from all files in {input_folder} is successfully converted and saved to {output_file}.")

# Example usage:
# To process specific files:
# convert_dat_to_csv("data/raw", "data/processed/magmeter_data.csv", selected_files=["AIDU-SIN.test1.DAT", "AIDU-SIN.test2.DAT"])
# To process all .DAT files in the folder:
convert_dat_to_csv("data/raw", "data/processed/magmeter_data.csv", selected_files=['4HD.031425.DAT'])

No duplicate columns found in the output.
Data from 4HD.031425.DAT is successfully converted and saved to data/processed/magmeter_data.csv.


## 2. Interactive Visualization

Using the processed CSV and boundary parameters (`plots_coordination.csv`), generate an interactive contour map to isolate anomalous magnetic fields within specific target parcels.

In [2]:
import pandas as pd
import numpy as np
from shapely.geometry import Point, Polygon
import plotly.graph_objects as go
from scipy.interpolate import griddata

def plot_magnetic_field_contour(plot_name="plot1", grid_resolution=50, top_percentile=100, preset_field_value=0):
    """
    Creates an interactive 2D contour plot using Plotly to visualize the magnetic field 
    strength within a specified parcel, with an option to filter out lower values by 
    replacing them with a preset field value.
    
    Process:
      1. Load magnetometer and parcel boundary data.
      2. Extract the parcel boundary for the specified plot.
      3. Filter magnetometer data points inside the parcel.
      4. Optionally replace measured field values below the top percentile threshold 
         with a preset value.
      5. Interpolate the (Longitude, Latitude, Field Strength) data onto a regular grid.
      6. Generate an interactive 2D contour plot with an overlay of the parcel boundary.
    
    Parameters:
      plot_name (str): Name of the parcel (e.g., "plot1").
      grid_resolution (int): Number of grid points along each axis for interpolation.
      top_percentile (int): Value between 0 and 100 indicating the percentage of data points 
                            (by magnetic field strength) to retain. Data points not in the top 
                            percentile are set to the preset_field_value.
      preset_field_value (float): The value to assign for data points falling below the 
                                  specified percentile threshold.
    """
    # --- 1. Load CSV data ---
    try:
        magnetometer_data = pd.read_csv('data/processed/magmeter_data.csv')
        plots_coordination = pd.read_csv('data/boundaries/plots_coordination.csv')
    except Exception as e:
        print("Error reading CSV files:", e)
        return

    # --- 2. Extract parcel boundary for the specified plot ---
    plot_coords = plots_coordination[plots_coordination['plot_name'] == plot_name]
    if plot_coords.empty:
        print(f"No parcel found with plot name: {plot_name}")
        return
    
    # Build a list of (longitude, latitude) tuples for the parcel boundary.
    polygon_points = list(zip(plot_coords['longitude'], plot_coords['latitude']))
    if polygon_points[0] != polygon_points[-1]:
        polygon_points.append(polygon_points[0])
    parcel_polygon = Polygon(polygon_points)

    # --- 3. Filter magnetometer data points inside the parcel ---
    def is_inside(row):
        return parcel_polygon.contains(Point(row['Longitude'], row['Latitude']))
    inside_mask = magnetometer_data.apply(is_inside, axis=1)
    data_inside = magnetometer_data[inside_mask]
    if data_inside.empty:
        print(f"No magnetometer data found inside {plot_name}.")
        return

    # --- 4. Extract coordinates and apply percentile-based filtering ---
    x = data_inside['Longitude'].values
    y = data_inside['Latitude'].values
    z = data_inside['Data'].values
    if top_percentile < 100:
        threshold = np.percentile(z, 100 - top_percentile)
        z = np.where(z < threshold, preset_field_value, z)

    # --- 5. Interpolate scattered data onto a regular grid ---
    xi = np.linspace(x.min(), x.max(), grid_resolution)
    yi = np.linspace(y.min(), y.max(), grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)
    grid_z = griddata(points=(x, y), values=z, xi=(grid_x, grid_y), method='linear')

    # --- 6. Create an interactive 2D contour plot ---
    contour_trace = go.Contour(
        x=xi,
        y=yi,
        z=grid_z,
        colorscale='Viridis',
        contours=dict(
            showlines=True,
            start=np.nanmin(grid_z),
            end=np.nanmax(grid_z),
            size=(np.nanmax(grid_z) - np.nanmin(grid_z)) / 10  # Adjust contour density as needed.
        ),
        colorbar=dict(title='Magnetic Field Strength'),
        name='Contour'
    )

    # Overlay the parcel boundary for context.
    boundary_trace = go.Scatter(
        x=[pt[0] for pt in polygon_points],
        y=[pt[1] for pt in polygon_points],
        mode='lines',
        line=dict(color='red', width=2),
        name='Parcel Boundary'
    )

    fig = go.Figure(data=[contour_trace, boundary_trace])
    fig.update_layout(
        title=f"Interactive Contour Plot of Magnetic Field Strength within {plot_name}",
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        legend=dict(x=0, y=1, xanchor='left', yanchor='top')
    )
    fig.show()

# Example usage:
if __name__ == '__main__':
    # In this example, only the top 85% of data points (by magnetic field strength) retain their values,
    # while the rest are set to a preset value (e.g., 0).
    plot_magnetic_field_contour(plot_name="plot4", grid_resolution=100, top_percentile=100, preset_field_value=37100)